# Tunisian Bac Math RAG — Validation Notebook

Direct testing of the RAG engine without Streamlit.
This notebook validates: retrieval quality, prompt compilation, generation correctness.

In [ ]:
import time
import json
from rag_engine import TunisianMathRAG, QueryResult

## 1. Initialize the RAG engine
This loads BGE-M3 embeddings, ChromaDB, and Vertex AI. Should take 30-60s on first run.

In [ ]:
engine = TunisianMathRAG()
print(f"Collection loaded: {engine.chunk_count} chunks")

## 2. Test queries

Define test cases covering different chapters and exercise types.
Each result captures retrieval scores, selected docs, timings, and the generated answer.

In [ ]:
TEST_QUERIES = [
    # (question, mode, expected_chapter_hint)
    ("Comment montrer qu'une suite est convergente ?", "coaching", "Suites"),
    ("Résoudre z² + 2z + 5 = 0 dans C", "correction", "Nombres complexes"),
    ("Calculer la limite de (sin x)/x quand x tend vers 0", "correction", "Limites"),
    ("Montrer que f est continue sur R", "coaching", "Continuité"),
    ("Déterminer la matrice inverse de A", "correction", "Matrices"),
]

In [ ]:
def display_result(result: QueryResult):
    """Pretty-print a QueryResult for notebook inspection."""
    print("=" * 70)
    print(f"QUESTION:  {result.question}")
    print(f"MODE:      {result.mode}")
    print(f"CASE:      {result.retrieval_case}")
    print(f"CONFIDENCE:{result.confidence}")
    print(f"TIMINGS:   retrieval={result.retrieval_time:.2f}s  "
          f"generation={result.generation_time:.2f}s  "
          f"total={result.total_time:.2f}s")
    print(f"\nSELECTED DOCS ({len(result.selected_docs)}):")
    for d in result.selected_docs:
        print(f"  Rank {d.rank} | dist={d.distance:.3f} | "
              f"type={d.metadata.get('type','')} | "
              f"chapter={d.metadata.get('chapter','')} | "
              f"sol={d.metadata.get('is_solution','')} | "
              f"file={d.metadata.get('filename','')}")
    print(f"\nANSWER:")
    print(result.answer[:2000] if result.answer else f"ERROR: {result.error}")
    print("=" * 70)
    return result

In [ ]:
# Run all test queries and collect results
results = []
for question, mode, hint in TEST_QUERIES:
    print(f"\n>>> Testing: {question[:60]}...")
    r = engine.query(question, mode=mode)
    display_result(r)
    results.append({
        "question": question,
        "mode": mode,
        "expected_chapter": hint,
        "retrieval_case": r.retrieval_case,
        "confidence": r.confidence,
        "num_docs": len(r.selected_docs),
        "best_distance": r.selected_docs[0].distance if r.selected_docs else None,
        "top_chapter": r.selected_docs[0].metadata.get("chapter", "") if r.selected_docs else "",
        "retrieval_time": round(r.retrieval_time, 3),
        "generation_time": round(r.generation_time, 3),
        "has_answer": bool(r.answer),
        "error": r.error,
    })
    print()

## 3. Results summary table

In [ ]:
# Summary as a simple table (no pandas dependency needed)
header = f"{'Question':<45} {'Case':>4} {'Conf':>6} {'Docs':>4} {'BestD':>6} {'TopChapter':<20} {'RetT':>5} {'GenT':>5}"
print(header)
print("-" * len(header))
for r in results:
    q = r['question'][:42] + '...' if len(r['question']) > 42 else r['question']
    bd = f"{r['best_distance']:.3f}" if r['best_distance'] is not None else 'N/A'
    print(f"{q:<45} {r['retrieval_case']:>4} {r['confidence']:>6} "
          f"{r['num_docs']:>4} {bd:>6} {r['top_chapter']:<20} "
          f"{r['retrieval_time']:>5.2f} {r['generation_time']:>5.2f}")

## 4. Retrieval-only tests (no LLM cost)

Test retrieval quality without calling Gemini. Useful for tuning thresholds.

In [ ]:
def test_retrieval_only(engine, query, n=5):
    """Test just the two-stage retrieval without generation."""
    selected, first_pass, second_pass, case = engine._two_stage_retrieve(query)
    print(f"Query: {query}")
    print(f"Case: {case} | First pass: {len(first_pass)} | Second pass: {len(second_pass)} | Selected: {len(selected)}")
    for d in selected[:n]:
        print(f"  dist={d.distance:.3f} type={d.metadata.get('type',''):15s} "
              f"ch={d.metadata.get('chapter',''):20s} sol={d.metadata.get('is_solution','')} "
              f"file={d.metadata.get('filename','')}")
        print(f"    Excerpt: {d.content[:150]}...")
    print()

# These calls are FREE (no Gemini API cost, only local embedding + ChromaDB)
test_retrieval_only(engine, "Déterminer le module et l'argument de z = 1 + i")
test_retrieval_only(engine, "Étudier la continuité de f en 0")
test_retrieval_only(engine, "Calculer une intégrale par parties")

## 5. Syllabus guard test

Verify that out-of-program methods are refused.

In [ ]:
# This should trigger a refusal or bac-compatible alternative
guard_test = engine.query(
    "Utilise la règle de L'Hôpital pour calculer la limite",
    mode="correction"
)
print("SYLLABUS GUARD TEST")
print("Question:", guard_test.question)
print("Answer preview:")
print(guard_test.answer[:1000] if guard_test.answer else f"Error: {guard_test.error}")

## 6. DB statistics (no API cost)

In [ ]:
# Quick metadata distribution check
from collections import Counter

# Sample 500 chunks to check metadata distribution
sample = engine.collection.get(limit=500, include=["metadatas"])
metas = sample["metadatas"]

type_dist = Counter(m.get("type", "unknown") for m in metas)
chapter_dist = Counter(m.get("chapter", "unknown") for m in metas)
sol_dist = Counter(m.get("is_solution", "unknown") for m in metas)

print(f"Total chunks in DB: {engine.chunk_count}")
print(f"\nType distribution (sample of {len(metas)}):")
for t, c in type_dist.most_common():
    print(f"  {t:20s}: {c}")
print(f"\nSolution distribution:")
for s, c in sol_dist.most_common():
    print(f"  is_solution={s:6s}: {c}")
print(f"\nChapter distribution (top 10):")
for ch, c in chapter_dist.most_common(10):
    print(f"  {ch:30s}: {c}")